# ARIA v3.0: 全自動區域受災衝擊評估系統（動態監測版）

## 📋 專案概述

ARIA v3.0 是一個整合即時雨量監測與避難所風險評估的智能系統，能夠回答「**現在哪裡最危險？**」這個關鍵問題。

### 🎯 核心功能
- **🌧️ 即時雨量監測**：整合 CWA API 與歷史颱風資料
- **🏠 避難所風險評估**：結合地形坡度、河川距離、高程等因子
- **🗺️ 動態風險地圖**：即時更新風險分級（URGENT/CRITICAL）
- **📊 災害模擬**：以鳳凰颱風 (2025) 作為壓力測試情境

### 📁 資料來源
- **向量資料**：Week 3-4 避難所資料、地形風險評估
- **雨量資料**：CWA O-A0002-001 API、鳳凰颱風歷史資料
- **地形資料**：花蓮、宜蘭 DEM 數值地形模型
- **邊界資料**：2025 年版權台灣行政區界

### 🏗️ 系統架構
1. **資料載入模組**：自動讀取多格式資料檔案
2. **地形處理模組**：DEM 載入、裁切、坡度計算
3. **地理篩選模組**：依縣市篩選避難所與測站
4. **風險評估模組**：動態計算災害風險等級
5. **視覺化模組**：互動式地圖與統計圖表

### 🚀 使用流程
1. 載入所有基礎資料（CSV、JSON、Shapefile、DEM）
2. 處理地形資料（花蓮、宜蘭 DEM 裁切）
3. 篩選目標區域（宜蘭縣 + 花蓮縣）
4. 整合雨量資料（LIVE/SIMULATION 模式）
5. 執行風險評估與視覺化

---

**🔧 技術棧**：Python + GeoPandas + Rasterio + Folium + Pandas  
**📊 輸出格式**：互動式地圖、統計報表、風險分級圖  
**⚡ 特色功能**：即時監測、歷史重演、壓力測試

## Cell 1 - 載入必要套件與Data

### 📁 載入資料項目
- **shelter** - 避難所資料 (CSV)
- **river** - 河川多邊形 (Shapefile)  
- **county** - 行政區界 (Shapefile)
- **shelter_Hualien_buffer_500m** - 花蓮避難所500m緩衝區

### 🗺️ 座標系統處理
- 轉換所有地理資料至 **EPSG:3826** (TWD97 / TM2 zone 121)
- 建立統一的空間參考框架

In [ ]:
import csv
import pandas as pd
import geopandas as gpd
import rioxarray as rxr
import os
import glob
import zipfile
import requests
import subprocess
import numpy as np
import pandas as pd
import rioxarray as rxr
import rasterio
import matplotlib.pyplot as plt
from affine import Affine
from rioxarray.merge import merge_arrays
from dotenv import load_dotenv
import json
from rasterstats import zonal_stats
from rasterio.transform import from_bounds

load_dotenv("./.env", override=True)

shelter = pd.read_csv("../Data/shelter_marked.csv")
river = gpd.read_file("../Data/riverpoly")
county = gpd.read_file("../Data/TOWN_MOI_1140318/Town_Moi_1140318.shp")
shelter_Hualien_buffer_500m = pd.read_csv("../Data/shelter_Hualien_buffer_500.csv")

gdf_shelter_Hualien_buffer_500m = gpd.GeoDataFrame(
    shelter_Hualien_buffer_500m,
    geometry=gpd.points_from_xy(shelter_Hualien_buffer_500m["經度"], shelter_Hualien_buffer_500m["緯度"]),
    crs="EPSG:4326"
).to_crs("EPSG:3826")

gdf_shelter = gpd.GeoDataFrame(
    shelter,
    geometry=gpd.points_from_xy(shelter["經度"], shelter["緯度"]),
    crs="EPSG:4326"
).to_crs("EPSG:3826")
river_3826 = river.to_crs("EPSG:3826")
county_3826 = county.to_crs("EPSG:3826")


## Cell 2 - 宜蘭與花蓮 DEM 處理流程

### 📋 處理步驟概覽
1. **📖 讀取 DTM 索引 CSV** - 2025年版全臺灣20公尺網格數值地形模型
2. **🎯 篩選宜蘭縣資料** - 從索引中提取宜蘭縣相關圖資
3. **⬇️ 下載宜蘭 DEM zip** - 從網路下載宜蘭縣 DEM 資料包
4. **📦 解壓縮檔案** - 解壓 DEM 資料至指定目錄
5. **🔍 找出 .grd 檔案** - 定位原始地形資料檔案
6. **🔄 GRD → TIF 轉換** - 使用 GDAL 將格式轉換為 TIF
7. **🛠️ 修正方向與 CRS** - 修正 upside-down 問題並設定 EPSG:3826
8. **🔗 合併宜蘭 TIF** - 將多個 TIF 檔案合併為單一 DEM
9. **✂️ 行政區裁切** - 使用 county_3826 裁切宜蘭縣範圍
10. **🏔️ 讀取花蓮 DEM** - 直接載入現有花蓮 DEM 檔案
11. **🎨 輸出 PNG 視覺圖** - 生成最終 DEM 視覺化成果

### 📁 關鍵路徑設定
```python
# 輸入資料
csv_path = "../Data/2025年版全臺灣20公尺網格數值地形模型DTM資料.csv"
hualien_dem_path = "../Data/dem_20m_hualien.tif"

# 處理目錄
download_dir = "../Data/yilan_dem"
output_dir = "../homework5/output"

# 最終輸出
yilan_clipped_tif = "../Data/yilan_dem/yilan_clipped_3826.tif"

In [ ]:
# =========================
# 0. 基本設定
# =========================
csv_path = "../Data/2025年版全臺灣20公尺網格數值地形模型DTM資料.csv"
download_dir = "../Data/yilan_dem"
extract_dir = os.path.join(download_dir, "extracted")
tif_dir = os.path.join(download_dir, "tif_tiles")
fixed_tif_dir = os.path.join(download_dir, "fixed_tif_tiles")
zip_path = os.path.join(download_dir, "yilan_dem.zip")
yilan_merged_tif = os.path.join(download_dir, "yilan_merged.tif")
yilan_clipped_tif = os.path.join(download_dir, "yilan_clipped_3826.tif")
hualien_dem_path = "../Data/dem_20m_hualien.tif"
output_dir = "../homework5/output"

os.makedirs(download_dir, exist_ok=True)
os.makedirs(extract_dir, exist_ok=True)
os.makedirs(tif_dir, exist_ok=True)
os.makedirs(fixed_tif_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

yilan_png_path = os.path.join(output_dir, "yilan_dem.png")
hualien_png_path = os.path.join(output_dir, "hualien_dem.png")

# =========================
# 1. 讀取 DTM 索引 CSV
# =========================
print("1. 讀取 DTM 索引 CSV")
dtm_index = pd.read_csv(csv_path)
print("CSV 欄位：", dtm_index.columns.tolist())

required_cols = ["圖資名稱", "連結網址"]
for col in required_cols:
    if col not in dtm_index.columns:
        raise KeyError(f"CSV 缺少必要欄位：{col}")

# =========================
# 2. 篩選宜蘭縣資料
# =========================
print("\n2. 篩選宜蘭縣資料")
yilan_row = dtm_index[dtm_index["圖資名稱"].astype(str).str.contains("宜蘭縣", na=False)]

print("宜蘭縣資料筆數：", len(yilan_row))
print(yilan_row[["圖資名稱", "連結網址"]])

if yilan_row.empty:
    raise ValueError("找不到宜蘭縣資料，請檢查 CSV 內容。")

yilan_url = yilan_row["連結網址"].iloc[0]
print("宜蘭下載連結：", yilan_url)

# =========================
# 3. 下載宜蘭 DEM zip
# =========================
print("\n3. 下載宜蘭 DEM zip")
if not os.path.exists(zip_path):
    print("開始下載...")
    r = requests.get(yilan_url, stream=True, timeout=120)
    r.raise_for_status()

    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    print("下載完成：", zip_path)
else:
    print("ZIP 已存在，略過下載：", zip_path)

# =========================
# 4. 解壓 zip
# =========================
print("\n4. 解壓 zip")
if len(os.listdir(extract_dir)) == 0:
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)
    print("解壓完成：", extract_dir)
else:
    print("解壓資料夾已有內容，略過解壓")

# =========================
# 5. 找出 .grd 檔
# =========================
print("\n5. 找出 .grd 檔")
grd_list = sorted(glob.glob(os.path.join(extract_dir, "**", "*.grd"), recursive=True))
print("找到的 .grd 數量：", len(grd_list))
print("前5個 .grd：", grd_list[:5])

if len(grd_list) == 0:
    raise FileNotFoundError("找不到 .grd 檔，請檢查解壓內容。")

# =========================
# 6. 用 GDAL 把 .grd 轉成 .tif
# =========================
print("\n6. .grd 轉 .tif")
converted_tifs = []

for grd_path in grd_list:
    base_name = os.path.splitext(os.path.basename(grd_path))[0]
    tif_path = os.path.join(tif_dir, f"{base_name}.tif")

    if not os.path.exists(tif_path):
        cmd = ["gdal_translate", grd_path, tif_path]
        print("轉檔中：", " ".join(cmd))
        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            print(f"轉檔失敗：{grd_path}")
            print(result.stderr)
            continue

    converted_tifs.append(tif_path)

print("成功轉出的 tif 數量：", len(converted_tifs))
print("前5個 tif：", converted_tifs[:5])

if len(converted_tifs) == 0:
    raise RuntimeError("所有 .grd 都無法轉成 .tif，請檢查 GDAL 是否安裝或支援此格式。")

# =========================
# 7. 修正 upside-down tif + 補 CRS
# =========================
print("\n7. 修正 tif 方向並補 CRS")
fixed_tifs = []

for tif_path in converted_tifs:
    base_name = os.path.basename(tif_path)
    out_path = os.path.join(fixed_tif_dir, base_name)

    if os.path.exists(out_path):
        fixed_tifs.append(out_path)
        continue

    with rasterio.open(tif_path) as src:
        data = src.read(1)
        meta = src.meta.copy()
        transform = src.transform

        meta["crs"] = "EPSG:3826"

        if transform.e > 0:
            print("修正方向：", base_name)

            data = np.flipud(data)

            new_transform = Affine(
                transform.a,
                transform.b,
                transform.c,
                transform.d,
                -transform.e,
                transform.f + transform.e * src.height
            )

            meta.update({
                "transform": new_transform,
                "height": data.shape[0],
                "width": data.shape[1]
            })
        else:
            meta.update({"transform": transform})

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(data, 1)

    fixed_tifs.append(out_path)

print("修正後 tif 數量：", len(fixed_tifs))
print("前5個修正後 tif：", fixed_tifs[:5])

if len(fixed_tifs) == 0:
    raise RuntimeError("沒有可用的修正後 tif。")

# =========================
# 8. merge 宜蘭 tif
# =========================
print("\n8. merge 宜蘭 tif")
yilan_tiles = []

for path in fixed_tifs:
    da = rxr.open_rasterio(path, masked=True).squeeze()

    if da.rio.crs is None:
        da = da.rio.write_crs("EPSG:3826")

    yilan_tiles.append(da)

yilan_dem = merge_arrays(yilan_tiles)
yilan_dem = yilan_dem.where((yilan_dem > -100) & (yilan_dem < 5000))

print("宜蘭合併 DEM：")
print(yilan_dem)
print("宜蘭 CRS:", yilan_dem.rio.crs)
print("宜蘭 bounds:", yilan_dem.rio.bounds())

yilan_dem.rio.to_raster(yilan_merged_tif)
print("已輸出宜蘭 merged tif：", yilan_merged_tif)

# =========================
# 9. 用 county_3826 裁切宜蘭
# 你必須先在前面的 notebook 中已有 county_3826
# =========================
print("\n9. 使用 county_3826 裁切宜蘭")

print("county_3826 欄位：", county_3826.columns.tolist())
print("county_3826 CRS:", county_3826.crs)

name_col = None
for c in county_3826.columns:
    c_str = str(c)
    if "COUNTY" in c_str.upper() or "縣市" in c_str or "NAME" in c_str.upper():
        name_col = c
        break

if name_col is None:
    raise ValueError("找不到縣市名稱欄位，請先檢查 county_3826.columns。")

yilan_boundary = county_3826[county_3826[name_col].astype(str).str.contains("宜蘭", na=False)]

if yilan_boundary.empty:
    raise ValueError("county_3826 中找不到宜蘭縣。")

yilan_boundary = yilan_boundary.dissolve()

if yilan_dem.rio.crs is None:
    raise ValueError("yilan_dem 沒有 CRS。")

if str(yilan_dem.rio.crs) != str(yilan_boundary.crs):
    yilan_boundary = yilan_boundary.to_crs(yilan_dem.rio.crs)

yilan_clipped = yilan_dem.rio.clip(
    yilan_boundary.geometry,
    yilan_boundary.crs,
    drop=True
)

yilan_clipped = yilan_clipped.where((yilan_clipped > -100) & (yilan_clipped < 5000))
yilan_clipped.rio.to_raster(yilan_clipped_tif)
print("已輸出宜蘭裁切 DEM：", yilan_clipped_tif)

# =========================
# 10. 讀取花蓮 DEM
# =========================
print("\n10. 讀取花蓮 DEM")

if not os.path.exists(hualien_dem_path):
    raise FileNotFoundError(f"找不到花蓮 DEM：{hualien_dem_path}")

hualien_dem = rxr.open_rasterio(hualien_dem_path, masked=True).squeeze()

if hualien_dem.rio.crs is None:
    hualien_dem = hualien_dem.rio.write_crs("EPSG:3826")
elif str(hualien_dem.rio.crs) != "EPSG:3826":
    hualien_dem = hualien_dem.rio.reproject("EPSG:3826")

hualien_dem = hualien_dem.where((hualien_dem > -100) & (hualien_dem < 5000))

print("花蓮 DEM：")
print(hualien_dem)
print("花蓮 CRS:", hualien_dem.rio.crs)
print("花蓮 bounds:", hualien_dem.rio.bounds())

# =========================
# 11. 輸出 PNG
# =========================
print("\n11. 輸出 PNG 到 output 資料夾")

# Yilan DEM PNG
fig, ax = plt.subplots(figsize=(10, 8))
img1 = yilan_clipped.plot(ax=ax, cmap="terrain", add_colorbar=True)
ax.set_title("Yilan DEM")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.savefig(yilan_png_path, dpi=300, bbox_inches="tight")
plt.close(fig)
print("已輸出：", yilan_png_path)

# Hualien DEM PNG
fig, ax = plt.subplots(figsize=(10, 8))
img2 = hualien_dem.plot(ax=ax, cmap="terrain", add_colorbar=True)
ax.set_title("Hualien DEM")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.savefig(hualien_png_path, dpi=300, bbox_inches="tight")
plt.close(fig)
print("已輸出：", hualien_png_path)

## Cell 3 - 篩選宜蘭縣避難所

### 🗺️ 做什麼
從全台灣的避難所資料中，把位於宜蘭縣範圍內的避難所篩選出來。

### 🔧 怎麼做
1. 先從縣市邊界資料中找出宜蘭縣的邊界線
2. 用空間連結 (spatial join) 的方式，找出完全在宜蘭縣界內的避難所
3. 把結果存成 `shelter_Yilan_true` 這個變數

### 📊 結果
會得到所有位於宜蘭縣內的避難所資料，可以用來做後續的風險分析。

In [ ]:
# 先從 county_3826 篩出宜蘭縣邊界
yilan_boundary = county_3826[
    county_3826["COUNTYNAME"].astype(str).str.contains("宜蘭", na=False)
].dissolve()

# 用 spatial join 篩出宜蘭縣內的 shelter
shelter_Yilan_true = gpd.sjoin(
    gdf_shelter,
    yilan_boundary[["geometry"]],
    how="inner",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

print("宜蘭 shelter 數量：", len(shelter_Yilan_true))


## Cell 4 - 宜蘭避難所地形分析

### 🎯 做什麼
分析宜蘭縣內每個避難所周圍 500 公尺範圍的地形特性。

### 📊 分析項目
1. **建立 500m 緩衝區** - 每個避難所周圍 500 公尺圓形區域
2. **計算高程統計** - 平均高程、標準差
3. **計算坡度分析** - 最大坡度、坡度分佈
4. **生成視覺圖** - DEM 高程圖與坡度圖
5. **輸出分析結果** - 包含地形特性的避難所 CSV

### 🏗️ 技術方法
- 使用 rasterstats 進行區域統計 (zonal statistics)
- 20 公尺解析度的 DEM 資料
- 坡度計算基於數值梯度方法
- 排除不合理的高程值 (-100m ~ 5000m)

### 📈 預期結果
每個避難所會得到：
- mean_elev (平均高程)
- std_elev (高程標準差)  
- max_slope (最大坡度)

In [ ]:


# =========================
# Cell 2: 宜蘭 shelter 做地形 buffer 分析
# =========================

output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)

# 1. 建立 500m shelter buffer
shelter_Yilan_buffer_500m = shelter_Yilan_true.copy()
shelter_Yilan_buffer_500m["geometry"] = shelter_Yilan_buffer_500m.geometry.buffer(500)

print("500m shelter buffers created.")
print(f"Number of shelter buffers: {len(shelter_Yilan_buffer_500m)}")

# 2. 從 DEM 取出 elevation array
if len(yilan_clipped.shape) == 3:
    elevation = yilan_clipped.values[0].astype("float64")
else:
    elevation = yilan_clipped.values.astype("float64")

nodata = yilan_clipped.rio.nodata
if nodata is not None:
    elevation[elevation == nodata] = np.nan

# 額外過濾不合理值
elevation[(elevation < -100) | (elevation > 5000)] = np.nan

# 3. 計算 slope (degree)
dy, dx = np.gradient(elevation, 20)  # 20m pixel spacing
slope_deg = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

# 4. 坡度摘要統計
slope_min = np.nanmin(slope_deg)
slope_max = np.nanmax(slope_deg)
slope_mean = np.nanmean(slope_deg)
slope_pct_gt30 = np.sum(slope_deg > 30) / np.sum(~np.isnan(slope_deg)) * 100

print("\n=== Yilan Slope Summary Statistics ===")
print(f"Minimum slope (deg): {slope_min:.2f}")
print(f"Maximum slope (deg): {slope_max:.2f}")
print(f"Mean slope (deg): {slope_mean:.2f}")
print(f"Percentage of pixels with slope > 30°: {slope_pct_gt30:.2f}%")

# 5. 視覺化 elevation 與 slope
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im1 = axes[0].imshow(elevation, cmap="terrain")
axes[0].set_title("Yilan DEM Elevation")
axes[0].axis("off")
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04, label="Elevation (m)")

im2 = axes[1].imshow(slope_deg, cmap="YlOrRd")
axes[1].set_title("Yilan Slope (degrees)")
axes[1].axis("off")
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04, label="Slope (deg)")

plt.tight_layout()
plt.show()

# 6. 準備 zonal statistics 的 raster transform
height, width = elevation.shape
left, bottom, right, top = yilan_clipped.rio.bounds()
transform = from_bounds(left, bottom, right, top, width, height)

# 7. Zonal stats：mean elevation, std elevation
elev_stats = zonal_stats(
    shelter_Yilan_buffer_500m,
    elevation,
    affine=transform,
    stats=["mean", "std"],
    nodata=np.nan
)

# 8. Zonal stats：max slope
slope_stats = zonal_stats(
    shelter_Yilan_buffer_500m,
    slope_deg,
    affine=transform,
    stats=["max"],
    nodata=np.nan
)

# 9. 回填欄位
shelter_Yilan_buffer_500m["mean_elev"] = [s["mean"] for s in elev_stats]
shelter_Yilan_buffer_500m["std_elev"] = [s["std"] for s in elev_stats]
shelter_Yilan_buffer_500m["max_slope"] = [s["max"] for s in slope_stats]

#10. drop欄位

cols_to_keep = [
    "序號",
    "村里",
    "避難收容處所地址",
    "經度",
    "緯度",
    "避難收容處所名稱",
    "預計收容村里",
    "geometry",
    "mean_elev",
    "std_elev",
    "max_slope"
]
gdf_shelter_Yilan_buffer_500m = shelter_Yilan_buffer_500m[cols_to_keep]
# 10. 檢查前五列
print("\n=== Updated shelter_Yilan_buffer_500m (first 5 rows) ===")
display(shelter_Yilan_buffer_500m.head())
print(gdf_shelter_Yilan_buffer_500m.columns.tolist())

# 11. 匯出 CSV
# CSV 不能直接存 shapely geometry，所以先轉成 WKT 字串
yilan_export = gdf_shelter_Yilan_buffer_500m.copy()
yilan_export["geometry"] = yilan_export["geometry"].astype(str)

csv_out = os.path.join(output_dir, "shelter_Yilan_buffer_500m.csv")
yilan_export.to_csv(csv_out, index=False, encoding="utf-8-sig")

print(f"\n宜蘭版本 CSV 已輸出：{csv_out}")

## Cell 5 - 合併花蓮與宜蘭避難所

### 🎯 做什麼
把花蓮縣和宜蘭縣的避難所 500m 緩衝分析結果合併成一個完整的資料集。

### 📊 合併內容
- **花蓮避難所** - `gdf_shelter_Hualien_buffer_500m`
- **宜蘭避難所** - `gdf_shelter_Yilan_buffer_500m`
- **統一變數** - `gdf_shelter_buffer_500m_all`

### 📈 預期結果
會得到包含花蓮、宜蘭兩縣所有避難所的完整資料集，每個避難所都有：
- 基本資料 (地址、名稱、座標)
- 地形分析 (平均高程、標準差、最大坡度)
- 空間緩衝區 (500m 圓形範圍)


In [ ]:
gdf_shelter_buffer_500m_all = gpd.GeoDataFrame(
    pd.concat(
        [gdf_shelter_Hualien_buffer_500m, gdf_shelter_Yilan_buffer_500m],
        ignore_index=True,
    ),
    crs="EPSG:3826"
)

print("合併後筆數：", len(gdf_shelter_buffer_500m_all))
display(gdf_shelter_buffer_500m_all.head())
print(gdf_shelter_buffer_500m_all.columns.tolist())

## Cell 6 - 河流風險區分析

### 🎯 做什麼
為宜蘭和花蓮縣建立河流風險緩衝區，分為高風險區和中等風險區兩個層級。

### 🌊 分析流程
1. **縣市邊界提取** - 分別建立宜蘭、花蓮縣界
2. **河流裁切** - 用 overlay 取出各縣內的河流段
3. **雙層緩衝建置**：
   - 高風險區：河流周圍 500m (BUFFER_HIGH)
   - 中等風險區：500m-1000m 環狀區 (BUFFER_MID)
4. **風險區標記** - 為每個區域標記風險等級和距離範圍

### 📊 輸出變數
- `rivers_buffer_500_Yilan` - 宜蘭高風險區
- `rivers_ring_500_1000_Yilan` - 宜蘭中等風險區
- `rivers_buffer_Yilan` - 宜蘭完整風險區
- `rivers_buffer_500_Hualien` - 花蓮高風險區
- `rivers_ring_500_1000_Hualien` - 花蓮中等風險區
- `rivers_buffer_Hualien` - 花蓮完整風險區
- `rivers_buffer_all` - 兩縣合併風險

In [ ]:
# 直接指定縣市名稱欄位
# 如果不是 COUNTYNAME，請改成你的實際欄位名
name_col = "COUNTYNAME"

# 宜蘭邊界
yilan_boundary = county_3826[
    county_3826[name_col].astype(str).str.contains("宜蘭", na=False)
].copy().dissolve()

# 花蓮邊界
hualien_boundary = county_3826[
    county_3826[name_col].astype(str).str.contains("花蓮", na=False)
].copy().dissolve()

# 切出宜蘭河流
rivers_Yilan = gpd.overlay(river_3826, yilan_boundary[["geometry"]], how="intersection")
rivers_Yilan = gpd.GeoDataFrame(
    rivers_Yilan[["geometry"]].copy(),
    geometry="geometry",
    crs="EPSG:3826"
)

# 切出花蓮河流
rivers_Hualien = gpd.overlay(river_3826, hualien_boundary[["geometry"]], how="intersection")
rivers_Hualien = gpd.GeoDataFrame(
    rivers_Hualien[["geometry"]].copy(),
    geometry="geometry",
    crs="EPSG:3826"
)

print("宜蘭河流筆數：", len(rivers_Yilan))
print("花蓮河流筆數：", len(rivers_Hualien))

display(rivers_Yilan.head())
display(rivers_Hualien.head())

# 讀取 homework5/.env
env_path = "./.env"
load_dotenv(env_path, override=True)

BUFFER_HIGH = int(os.getenv("BUFFER_HIGH"))
BUFFER_MID = int(os.getenv("BUFFER_MID"))

print(f"BUFFER_HIGH = {BUFFER_HIGH}")
print(f"BUFFER_MID = {BUFFER_MID}")

def build_river_buffers(rivers_gdf, county_name, buffer_high, buffer_mid):
    # 0-buffer_high
    rivers_buffer_500 = rivers_gdf.copy()
    rivers_buffer_500["geometry"] = rivers_buffer_500.geometry.buffer(buffer_high)

    # 0-buffer_mid
    rivers_buffer_1000 = rivers_gdf.copy()
    rivers_buffer_1000["geometry"] = rivers_buffer_1000.geometry.buffer(buffer_mid)

    # dissolve
    rivers_buffer_500_diss = rivers_buffer_500.dissolve().reset_index(drop=True)
    rivers_buffer_500_diss["risk_zone"] = "high"
    rivers_buffer_500_diss["dist_min"] = 0
    rivers_buffer_500_diss["dist_max"] = buffer_high
    rivers_buffer_500_diss["county"] = county_name

    rivers_buffer_1000_diss = rivers_buffer_1000.dissolve().reset_index(drop=True)

    # ring
    rivers_ring_500_1000 = rivers_buffer_1000_diss.copy()
    rivers_ring_500_1000["geometry"] = rivers_buffer_1000_diss.geometry.difference(
        rivers_buffer_500_diss.iloc[0].geometry
    )
    rivers_ring_500_1000["risk_zone"] = "mid"
    rivers_ring_500_1000["dist_min"] = buffer_high
    rivers_ring_500_1000["dist_max"] = buffer_mid
    rivers_ring_500_1000["county"] = county_name

    # merge
    rivers_buffer = gpd.GeoDataFrame(
        pd.concat([rivers_buffer_500_diss, rivers_ring_500_1000], ignore_index=True),
        geometry="geometry",
        crs=rivers_gdf.crs
    )

    print(f"{county_name} rivers_buffer 筆數:", len(rivers_buffer))
    return rivers_buffer_500_diss, rivers_ring_500_1000, rivers_buffer

rivers_buffer_500_Yilan, rivers_ring_500_1000_Yilan, rivers_buffer_Yilan = build_river_buffers(
    rivers_Yilan, "Yilan", BUFFER_HIGH, BUFFER_MID
)

rivers_buffer_500_Hualien, rivers_ring_500_1000_Hualien, rivers_buffer_Hualien = build_river_buffers(
    rivers_Hualien, "Hualien", BUFFER_HIGH, BUFFER_MID
)

rivers_buffer_all = gpd.GeoDataFrame(
    pd.concat([rivers_buffer_Yilan, rivers_buffer_Hualien], ignore_index=True),
    geometry="geometry",
    crs="EPSG:3826"
)

display(rivers_buffer_all)
print(rivers_buffer_all.columns.tolist())

## Cell 7 - 綜合風險評估

### 🎯 做什麼
為宜蘭和花蓮縣的所有避難所進行綜合風險評估，結合河川距離、地形坡度、高程等多個因子。

### 📊 評估因子
1. **河川距離分類** - <500m、500~1000m、>1000m
2. **地形坡度閾值** - SLOPE_THRESHOLD (從 .env 讀取)
3. **高程安全值** - ELEVATION_LOW (從 .env 讀取)
4. **多因子綜合** - 組合不同條件進行風險分級

### ⚡ 風險等級定義
- **low** - 基礎安全條件
- **medium** - 距離河川 500~1000m 或高程過低
- **high** - 距離河川 <500m 或坡度過陡
- **extremely_high** - 距離河川 <500m 且坡度過陡

### 🏗️ 評估邏輯
```python
# 條件判斷
cond_lt_1000 = 距離河川500~1000m 或 高程過低
cond_lt_500 = 距離河川<500m 或 坡度過陡
cond_steep = 坡度 > SLOPE_THRESHOLD
cond_low_elev = 高程 < ELEVATION_LOW

# 風險等級分配
if 距離河川<500m and 坡度過陡 → extremely_high
elif 距離河川<500m or 坡度過陡 → high
elif 距離河川500~1000m or 高程過低 → medium
else → low

In [ ]:
# ---------------------------------------------------------
# 1. 讀取 .env 參數（homework5/.env）
# ---------------------------------------------------------
env_path = "./.env"
load_dotenv(env_path, override=True)

slope_threshold_raw = os.getenv("SLOPE_THRESHOLD")
elevation_low_raw = os.getenv("ELEVATION_LOW")

print("SLOPE_THRESHOLD raw =", slope_threshold_raw)
print("ELEVATION_LOW raw =", elevation_low_raw)

if slope_threshold_raw is None or elevation_low_raw is None:
    raise ValueError("讀不到 SLOPE_THRESHOLD 或 ELEVATION_LOW，請檢查 .env")

SLOPE_THRESHOLD = float(slope_threshold_raw)
ELEVATION_LOW = float(elevation_low_raw)

print("SLOPE_THRESHOLD =", SLOPE_THRESHOLD)
print("ELEVATION_LOW =", ELEVATION_LOW)

# ---------------------------------------------------------
# 2. 以 gdf_shelter_buffer_500m_all 為基底複製新資料
# ---------------------------------------------------------
gdf_shelter_buffer_500_risk_all = gdf_shelter_buffer_500m_all.copy()

# ---------------------------------------------------------
# 4. 從 rivers_buffer_all 拆出 high / mid 區
# ---------------------------------------------------------
rivers_buffer_high = rivers_buffer_all[
    rivers_buffer_all["risk_zone"] == "high"
].copy()

rivers_buffer_mid = rivers_buffer_all[
    rivers_buffer_all["risk_zone"] == "mid"
].copy()

print("\n河川 buffer 統計：")
print("high 區筆數:", len(rivers_buffer_high))
print("mid 區筆數:", len(rivers_buffer_mid))

# ---------------------------------------------------------
# 5. 用避難所點位判斷河川距離分類
#    注意：距離分類應該用 shelter 點位，不是 500m shelter polygon
# ---------------------------------------------------------
point_df = gdf_shelter_buffer_500m_all.copy().reset_index(drop=True)
point_df["__sid"] = point_df.index

gdf_shelter_buffer_500_risk_all = gdf_shelter_buffer_500_risk_all.reset_index(drop=True)
gdf_shelter_buffer_500_risk_all["__sid"] = gdf_shelter_buffer_500_risk_all.index

# 先建立三個輔助欄位
point_df["距離河川500m以內"] = 0
point_df["距離河川500-1000m"] = 0
point_df["距離河川超過1000m"] = 0

# 判斷 <500m
join_high = gpd.sjoin(
    point_df[["__sid", "geometry"]],
    rivers_buffer_high[["geometry"]],
    how="inner",
    predicate="within"
)
high_ids = set(join_high["__sid"].unique())
point_df.loc[point_df["__sid"].isin(high_ids), "距離河川500m以內"] = 1

# 判斷 500~1000m
join_mid = gpd.sjoin(
    point_df[["__sid", "geometry"]],
    rivers_buffer_mid[["geometry"]],
    how="inner",
    predicate="within"
)
mid_ids = set(join_mid["__sid"].unique())
point_df.loc[point_df["__sid"].isin(mid_ids), "距離河川500-1000m"] = 1

# 其餘 >1000m
outside_mask = (
    (point_df["距離河川500m以內"] == 0) &
    (point_df["距離河川500-1000m"] == 0)
)
point_df.loc[outside_mask, "距離河川超過1000m"] = 1

# merge 回 shelter buffer 表
distance_cols = [
    "__sid",
    "距離河川500m以內",
    "距離河川500-1000m",
    "距離河川超過1000m"
]

gdf_shelter_buffer_500_risk_all = gdf_shelter_buffer_500_risk_all.merge(
    point_df[distance_cols],
    on="__sid",
    how="left"
)

# ---------------------------------------------------------
# 6. 整合成單一欄位 river_distance_category
# ---------------------------------------------------------
gdf_shelter_buffer_500_risk_all["river_distance_category"] = ">1000m"

gdf_shelter_buffer_500_risk_all.loc[
    gdf_shelter_buffer_500_risk_all["距離河川500-1000m"] == 1,
    "river_distance_category"
] = "500~1000m"

gdf_shelter_buffer_500_risk_all.loc[
    gdf_shelter_buffer_500_risk_all["距離河川500m以內"] == 1,
    "river_distance_category"
] = "<500m"

# ---------------------------------------------------------
# 7. 若高程/坡度缺值，先補安全值避免誤判
# ---------------------------------------------------------
gdf_shelter_buffer_500_risk_all["max_slope"] = gdf_shelter_buffer_500_risk_all["max_slope"].fillna(-9999)
gdf_shelter_buffer_500_risk_all["mean_elev"] = gdf_shelter_buffer_500_risk_all["mean_elev"].fillna(999999)

# ---------------------------------------------------------
# 8. 建立風險等級條件
# ---------------------------------------------------------
cond_lt_500 = gdf_shelter_buffer_500_risk_all["river_distance_category"] == "<500m"
cond_lt_1000 = gdf_shelter_buffer_500_risk_all["river_distance_category"].isin(["<500m", "500~1000m"])
cond_steep = gdf_shelter_buffer_500_risk_all["max_slope"] > SLOPE_THRESHOLD
cond_low_elev = gdf_shelter_buffer_500_risk_all["mean_elev"] < ELEVATION_LOW

# ---------------------------------------------------------
# 9. 新增風險等級欄位
# ---------------------------------------------------------
gdf_shelter_buffer_500_risk_all["risk_level"] = "low"

gdf_shelter_buffer_500_risk_all.loc[
    cond_lt_1000 & cond_low_elev,
    "risk_level"
] = "medium"

gdf_shelter_buffer_500_risk_all.loc[
    cond_lt_500 | cond_steep,
    "risk_level"
] = "high"

gdf_shelter_buffer_500_risk_all.loc[
    cond_lt_500 & cond_steep,
    "risk_level"
] = "extremely_high"

# ---------------------------------------------------------
# 10. 清理暫時欄位
# ---------------------------------------------------------
gdf_shelter_buffer_500_risk_all = gdf_shelter_buffer_500_risk_all.drop(
    columns=[
        "__sid",
        "距離河川500m以內",
        "距離河川500-1000m",
        "距離河川超過1000m"
    ],
    errors="ignore"
)

# ---------------------------------------------------------
# 11. 檢查結果
# ---------------------------------------------------------
print("\nriver_distance_category 統計：")
print(gdf_shelter_buffer_500_risk_all["river_distance_category"].value_counts(dropna=False))

print("\n風險等級統計：")
print(gdf_shelter_buffer_500_risk_all["risk_level"].value_counts(dropna=False))

print("\n前五筆資料：")
display(gdf_shelter_buffer_500_risk_all.head())

# ---------------------------------------------------------
# 12. 輸出新檔案：gdf_buffer_500_risk_all.csv
# ---------------------------------------------------------
output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)

csv_path = f"{output_dir}/gdf_shelter_buffer_500_risk_all.csv"

export_df = gdf_shelter_buffer_500_risk_all.copy()
export_df["geometry"] = export_df["geometry"].astype(str)

export_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"\nCSV 已輸出：{csv_path}")

## Cell 8 - CWA 即時雨量 API

### 🌧 做什麼
建立一個函數來呼叫中央氣象署的即時雨量 API，取得全台灣雨量測站資料。

### 📡 API 資訊
- **API 端點**：`https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001`
- **資料類型**：自動雨量站即時觀測資料
- **認證方式**：使用 API 金鑰進行授權
- **回傳格式**：JSON 格式的測站資料

### 🛡️ 錯誤處理機制
- **網路連線問題** - ConnectionError、Timeout
- **HTTP 狀態碼** - 401(認證失敗)、403(存取拒絕)、404(不存在)
- **API 回應檢查** - 確認 success 欄位
- **未預期錯誤** - 捕捉所有其他例外

### 📊 輸出結果
- **成功時**：回傳完整的 JSON 資料物件
- **失敗時**：回傳 None 並顯示錯誤訊息
- **測站數量**：顯示成功取得的測站數量

### 🎯 使用方式
```python
# 呼叫 API
api_key = "your-api-key-here"
weather_data = fetch_cwa_api(api_key)

if weather_data:
    print("成功取得即時雨量資料")
else:
    print("API 呼叫失敗")

In [ ]:
# Cell [2]: YOUR CODE HERE
# Write a function fetch_cwa_api(api_key) that:
# 1. Calls https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001
# 2. Returns JSON response
# 3. Handles errors with try/except

import requests
from typing import Optional, Dict, Any

def fetch_cwa_api(api_key: str) -> Optional[Dict[str, Any]]:
    """
    呼叫 CWA 雨量 API (O-A0002-001) 並取得即時資料
    
    Args:
        api_key (str): CWA 授權碼 (Authorization Key)
        
    Returns:
        Optional[Dict[str, Any]]: 
            - 成功時回傳 API 回應的 JSON 資料 (dict)
            - 失敗時回傳 None
    """
    
    # API 端點
    base_url = "https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001"
    
    # 請求參數
    params = {
        'Authorization': api_key,
        'format': 'JSON'
    }
    
    try:
        # 發送 GET 請求
        response = requests.get(base_url, params=params, timeout=10)
        
        # 檢查 HTTP 狀態碼
        response.raise_for_status()
        
        # 解析 JSON 回應
        data = response.json()
        
        # 檢查 API 回應狀態
        if data.get("success") == "true":
            stations = data.get("records", {}).get("Station", [])
            print(f"✅ 成功取得 {len(stations)} 個雨量測站資料")
            return data
        else:
            print(f"❌ API 回應失敗")
            return None
            
    except requests.exceptions.RequestException as e:
        # 處理網路相關錯誤
        if isinstance(e, requests.exceptions.ConnectionError):
            print("❌ 網路連線失敗")
        elif isinstance(e, requests.exceptions.Timeout):
            print("❌ 請求超時")
        elif isinstance(e, requests.exceptions.HTTPError):
            if e.response.status_code == 401:
                print("❌ 認證失敗，請檢查 API 金鑰")
            elif e.response.status_code == 403:
                print("❌ 存取被拒絕")
            elif e.response.status_code == 404:
                print("❌ API 端點不存在")
            else:
                print(f"❌ HTTP 錯誤 {e.response.status_code}")
        else:
            print(f"❌ 請求錯誤")
        
        return None
        
    except Exception as e:
        print(f"❌ 未預期錯誤: {str(e)}")
        return None

## Cell 9 - 雨量資料解析函數

### 🌧 做什麼
建立兩個函數來解析 CWA 雨量 API 的 JSON 回應，轉換為可用的地理資料格式。

### 📋 函數說明

#### `normalize_cwa_json(raw)`
- **目的**：檢測 JSON 格式並標準化測站列表
- **支援格式**：
  - CWA API 格式：`records.Station[]`
  - CWA XML 格式：`cwaopendata.dataset.Station[]`
- **輸出**：統一的測站列表

#### `parse_rainfall_json(data)`
- **目的**：將測站資料轉換為 GeoDataFrame
- **處理項目**：測站名稱、座標、雨量資料、行政區資訊
- **座標處理**：優先使用 WGS84，支援多種座標格式

### 🗺️ 座標處理邏輯
1. **WGS84 優先** - 尋找 CoordinateName == 'WGS84' 的座標
2. **多格式支援** - 處理 list 和 dict 兩種座標格式
3. **錯誤容錯** - 單站錯誤不影響整體處理
4. **座標驗證** - 檢查經緯度合理性（台灣範圍）

### 🌧 雨量資料處理
- **時段資料**：1小時、3小時、24小時累積雨量
- **NoData 處理**：將 -998 轉換為 0.0
- **型別轉換**：安全地轉換字串為數值
- **資料清理**：自動處理缺失值和異常值

### 📊 輸出結果
- **GeoDataFrame**：包含完整測站資料和幾何圖形
- **座標系統**：EPSG:4326 (WGS84)
- **資料欄位**：StationName、CountyName、rain_1hr、rain_3hr、rain_24hr

In [ ]:
from shapely.geometry import Point

def normalize_cwa_json(raw):
    if raw is None:
        print("❌ raw is None")
        return []

    if 'records' in raw and 'Station' in raw['records']:
        return raw['records']['Station']

    if 'cwaopendata' in raw:
        dataset = raw['cwaopendata'].get('dataset', {})
        if 'Station' in dataset:
            return dataset['Station']

    print("❌ 無法識別 JSON 格式")
    return []


def parse_rainfall_json(data):
    stations = normalize_cwa_json(data)
    print(f"stations len = {len(stations)}")

    if not stations:
        print("❌ 沒有測站資料")
        return gpd.GeoDataFrame({'geometry': []}, geometry='geometry', crs='EPSG:4326')

    rows = []

    def to_float(v):
        try:
            v = float(v)
            return 0.0 if v == -998 else v
        except:
            return 0.0

    def safe_extract_coordinates(coords, station_name, idx):
        """安全提取座標，支援多種格式但保持原有邏輯"""
        lat, lon = None, None
        coord_system = "Unknown"

        if isinstance(coords, list) and len(coords) >= 1:
            # 優先尋找WGS84座標（如果有標示的話）
            wgs84_coord = None
            for coord in coords:
                if coord.get('CoordinateName') == 'WGS84':
                    wgs84_coord = coord
                    break
            
            # 如果找到WGS84就用它，否則用第一個（保持原有邏輯）
            c = wgs84_coord if wgs84_coord else coords[0]
            coord_system = c.get('CoordinateName', 'Unknown')
            
            # 安全地提取經緯度
            try:
                lat_str = c.get('StationLatitude', '0')
                lon_str = c.get('StationLongitude', '0')
                lat = float(lat_str) if lat_str and lat_str != '0' else None
                lon = float(lon_str) if lon_str and lon_str != '0' else None
            except ValueError:
                if idx < 5:
                    print(f"⚠️ {station_name} 座標格式錯誤，跳過此測站")
                return None, None, "Format Error"
                
        elif isinstance(coords, dict):
            # 處理字典格式的座標
            coord_system = "Dict Format"
            try:
                lat_str = coords.get('StationLatitude', '0')
                lon_str = coords.get('StationLongitude', '0')
                lat = float(lat_str) if lat_str and lat_str != '0' else None
                lon = float(lon_str) if lon_str and lon_str != '0' else None
            except ValueError:
                if idx < 5:
                    print(f"⚠️ {station_name} 座標格式錯誤，跳過此測站")
                return None, None, "Format Error"

        # 驗證座標合理性（可選，不影響原有邏輯）
        if lat is not None and lon is not None:
            if not (21.8 <= lat <= 25.3 and 119.5 <= lon <= 122.5):
                if idx < 3:  # 只報告前3個異常座標
                    print(f"📍 {station_name} 座標可能異常: ({lat}, {lon}) [{coord_system}]")
                # 仍然使用這個座標，不跳過（保持原有行為）

        return lat, lon, coord_system

    for idx, station in enumerate(stations):
        try:
            name = station.get('StationName')
            sid = station.get('StationId')

            geo = station.get('GeoInfo', {})
            county = geo.get('CountyName')
            town = geo.get('TownName')
            coords = geo.get('Coordinates', [])

            # 使用改進的座標提取函數
            lat, lon, coord_system = safe_extract_coordinates(coords, name, idx)

            if lat is None or lon is None:
                if idx < 5:
                    print(f"⚠️ {name} 缺少有效座標")
                continue

            # 提取雨量資料（保持原有邏輯）
            rain = station.get('RainfallElement', {})
            r1 = to_float(rain.get('Past1hr', {}).get('Precipitation'))
            r3 = to_float(rain.get('Past3hr', {}).get('Precipitation'))
            r24 = to_float(rain.get('Past24hr', {}).get('Precipitation'))

            # 建立資料列（保持原有格式）
            rows.append({
                'StationName': name,
                'StationId': sid,
                'latitude': lat,
                'longitude': lon,
                'rain_1hr': r1,
                'rain_3hr': r3,
                'rain_24hr': r24,
                'CountyName': county,
                'TownName': town,
                'geometry': Point(lon, lat)
            })

            # 記錄座標系統資訊（僅供參考，不影響資料）
            if idx < 3:
                print(f"✅ {name} 使用 {coord_system} 座標: ({lat:.6f}, {lon:.6f})")

        except Exception as e:
            if idx < 5:
                print(f"⚠️ 第 {idx} 筆處理失敗: {e}")
            continue

    print(f"成功轉成 rows 的筆數 = {len(rows)}")

    if not rows:
        print("❌ 沒有任何測站成功轉成 geometry")
        return gpd.GeoDataFrame({'geometry': []}, geometry='geometry', crs='EPSG:4326')

    gdf = gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')
    print(f"✅ 載入 {len(gdf)} 個測站")
    return gdf

## Cell 10 - 檢查運行模式
### 確認當前app_mode狀態 

In [ ]:
print(os.getenv("APP_MODE"))

## Cell 11 - 模式切換

### 🎯 功能
根據 APP_MODE 自動載入即時或模擬雨量資料。

### 📋 模式
- **LIVE** - CWA API 即時資料
- **SIMULATION** - 鳳凰颱風歷史資料

### 🔄 處理
1. 讀取 APP_MODE
2. 載入對應資料
3. 轉換座標系統
4. 輸出結果

In [ ]:
# 1. 讀取模式，預設 SIMULATION
app_mode = os.getenv("APP_MODE", "SIMULATION").upper()
print(f"🔧 Active mode: {app_mode}")

gdf_rainfall = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

# 2. 根據模式決定資料來源
if app_mode == "LIVE":
    print("🌐 Loading LIVE data from CWA API...")
    
    api_key = os.getenv("CWA_API_KEY")
    if not api_key:
        print("❌ CWA_API_KEY not found in environment")
    else:
        try:
            raw_data = fetch_cwa_api(api_key)
            gdf_rainfall = parse_rainfall_json(raw_data)
        except Exception as e:
            print(f"❌ Failed to fetch LIVE data: {e}")

elif app_mode == "SIMULATION":
    print("📂 Loading SIMULATION data from local JSON...")
    
    simulation_file = os.getenv(
        "SIMULATION_DATA",
        "../Data/scenarios/fungwong_202511.json"
    )
    
    print(f"📍 File path: {simulation_file}")
    print(f"📍 Current working directory: {os.getcwd()}")
    
    try:
        with open(simulation_file, "r", encoding="utf-8") as f:
            raw_data = json.load(f)
        gdf_rainfall = parse_rainfall_json(raw_data)
    except FileNotFoundError:
        print(f"❌ Simulation file not found: {simulation_file}")
    except json.JSONDecodeError as e:
        print(f"❌ JSON decode error: {e}")
    except Exception as e:
        print(f"❌ Failed to load simulation data: {e}")

else:
    print(f"❌ Unknown APP_MODE: {app_mode}")

gdf_rainfall_3826 = gdf_rainfall.to_crs("EPSG:3826")
# 3. 顯示結果
if not gdf_rainfall.empty:
    print(f"✅ Loaded {len(gdf_rainfall)} stations")
else:
    print("❌ No rainfall stations loaded")

gdf_rainfall_3826.head()


## Cell 12 - 高雨量站緩衝分析

### 🎯 功能
篩選大雨量測站並建立影響範圍的緩衝區。

### 📊 處理步驟
1. 讀取 RAINFALL_BUFFER 參數 (預設 5000m)
2. 排除 -998 缺值資料
3. 篩選 rain_1hr > 30 mm/hr 測站
4. 建立 5km 緩衝區
5. 輸出結果統計

### 🌧 關鍵參數
- **RAINFALL_BUFFER** - 緩衝距離 (公尺)
- **雨量閾值** - 30 mm/hr
- **座標系統** - EPSG:3826

### 📈 預期結果
- **heavy_rain** - 大雨量測站
- **heavy_rain_buffer_5km** - 影響範圍緩衝區

In [ ]:
from dotenv import load_dotenv
import os
import geopandas as gpd

# =========================
# 1. 讀取 .env
# =========================
load_dotenv("./.env", override=True)

rainfall_buffer_raw = os.getenv("RAINFALL_BUFFER", "5000")
rainfall_buffer = float(rainfall_buffer_raw)

print("RAINFALL_BUFFER =", rainfall_buffer)

# =========================
# 2. 複製原始雨量站資料
#    這裡假設你現在用的是 gdf_fungwong
# =========================
gdf_rainfall = gdf_rainfall.copy()

# =========================
# 3. 排除 rain_1hr = -998 的缺值資料
# =========================
gdf_rainfall = gdf_rainfall[gdf_rainfall["rain_1hr"] != -998].copy()

print("排除 -998 後筆數：", len(gdf_rainfall))

# =========================
# 4. 確保 CRS 正確，先設 4326 再轉 3826
# =========================
if gdf_rainfall.crs is None:
    gdf_rainfall = gdf_rainfall.set_crs("EPSG:4326")

gdf_rainfall_3826 = gdf_rainfall.to_crs("EPSG:3826")

print("轉換後 CRS：", gdf_rainfall_3826.crs)

# =========================
# 5. 篩出 rain_1hr > 30 的測站
# =========================
heavy_rain = gdf_rainfall_3826[gdf_rainfall_3826["rain_1hr"] > 30].copy()

print(f"rain_1hr > 30 mm/hr 的測站數量：{len(heavy_rain)}")
display(heavy_rain[["StationName", "StationId", "CountyName", "TownName", "rain_1hr", "geometry"]].head())

# =========================
# 6. 建立 5km buffer
# =========================
heavy_rain_buffer_5km = heavy_rain.copy()
heavy_rain_buffer_5km["geometry"] = heavy_rain_buffer_5km.geometry.buffer(rainfall_buffer)

print(f"{rainfall_buffer} m buffer 建立完成，筆數：{len(heavy_rain_buffer_5km)}")
display(heavy_rain_buffer_5km.head())

## Cell 13 - 避難所暴雨影響分析

### 🎯 功能
分析哪些避難所位於高雨量測站 5km 影響範圍內。

### 📊 處理步驟
1. 清理重複雨量欄位
2. 統一座標系統 (EPSG:3826)
3. 空間連結 (sjoin) - 避難所 vs 暴雨緩衝區
4. 多暴雨區取最大雨量
5. 合併回避難所資料
6. 輸出分析結果

### 🗺️ 空間分析邏輯
- **intersects** - 避難所與暴雨緩衝區相交
- **groupby max** - 同一避難所對多個暴雨區取最大值
- **left join** - 保留所有避難所，影響值補 0

### 📈 關鍵輸出
- **一小時降雨量** - 避難所對應的最大暴雨量
- **terrain_risk** - 原有地形風險等級
- **CSV 輸出** - 完整分析結果檔案

### 🔄 後續應用
可用來：
- 綜合地形與暴雨風險
- 建立即時監測指標
- 避難所優先級評估

In [ ]:
# =========================
# 0. 先清掉可能重複的雨量欄位
# =========================
gdf_shelter_buffer_500_risk_all = gdf_shelter_buffer_500_risk_all.drop(
    columns=[
        "rain_1hr", "rain_1hr_x", "rain_1hr_y",
        "一小時降雨量", "一小時降雨量_x", "一小時降雨量_y"
    ],
    errors="ignore"
)
gdf_shelter_buffer_risk_all = gdf_shelter_buffer_500_risk_all.copy()
# =========================
# 1. 確保 CRS 一致
# =========================
target_crs = "EPSG:3826"

if gdf_shelter_buffer_risk_all.crs is None:
    gdf_shelter_buffer_risk_all = gdf_shelter_buffer_risk_all.set_crs(target_crs)
elif str(gdf_shelter_buffer_risk_all.crs) != target_crs:
    gdf_shelter_buffer_risk_all = gdf_shelter_buffer_risk_all.to_crs(target_crs)

if heavy_rain_buffer_5km.crs is None:
    heavy_rain_buffer_5km = heavy_rain_buffer_5km.set_crs(target_crs)
elif str(heavy_rain_buffer_5km.crs) != target_crs:
    heavy_rain_buffer_5km = heavy_rain_buffer_5km.to_crs(target_crs)

# =========================
# 2. 建立 ID（避免多對多問題）
# =========================
gdf_shelter_buffer_risk_all = gdf_shelter_buffer_risk_all.reset_index(drop=True).copy()
gdf_shelter_buffer_risk_all["__sid"] = gdf_shelter_buffer_risk_all.index
# =========================
# 3. sjoin：找出和暴雨 buffer 相交的 shelter
# =========================
affected_join = gpd.sjoin(
    gdf_shelter_buffer_risk_all[["__sid", "geometry"]],
    heavy_rain_buffer_5km[["rain_1hr", "geometry"]],
    how="left",
    predicate="intersects"
)

print("sjoin筆數：", len(affected_join))
display(affected_join.head())

# =========================
# 4. 同一 shelter 若碰到多個暴雨 buffer，取最大 rain_1hr
# =========================
affected_rain = (
    affected_join.groupby("__sid", as_index=False)["rain_1hr"]
    .max()
    .rename(columns={"rain_1hr": "一小時降雨量"})
)

display(affected_rain.head())

# =========================
# 5. merge 回主表
# =========================
gdf_shelter_buffer_risk_all = gdf_shelter_buffer_risk_all.merge(
    affected_rain,
    on="__sid",
    how="left"
)

# 沒有相交到暴雨 buffer 的補 0
gdf_shelter_buffer_risk_all["一小時降雨量"] = (
    gdf_shelter_buffer_risk_all["一小時降雨量"].fillna(0)
)

# =========================
# 6. 檢查
# =========================
print(gdf_shelter_buffer_risk_all["一小時降雨量"].describe())
display(gdf_shelter_buffer_risk_all.head())

# =========================
# 7. 刪除暫時欄位
# =========================
gdf_shelter_buffer_risk_all = gdf_shelter_buffer_risk_all.drop(
    columns="__sid",
    errors="ignore"
)
gdf_shelter_buffer_risk_all = gdf_shelter_buffer_risk_all.rename(
    columns={"risk_level": "terrain_risk"}
)
# =========================
# 8. 輸出 CSV
# =========================
output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)

csv_path = f"{output_dir}/gdf_shelter_buffer_risk.csv"

export_df = gdf_shelter_buffer_risk_all.copy()
export_df["geometry"] = export_df["geometry"].astype(str)

export_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"✅ 已輸出：{csv_path}")

## Cell 14 - 綜合風險評估

### 🎯 功能
結合暴雨量與地形風險，進行最終的綜合風險分級。

### 📊 風險等級定義
- **SAFE** - 基礎安全條件
- **WARNING** - 雨量 > 40mm 或地形高風險
- **URGENT** - 雨量 > 40mm 且地形高風險
- **CRITICAL** - 雨量 > 80mm

### 🧮 評估邏輯
```python
# 條件判斷
cond_rain_40 = 一小時降雨量 > 40
cond_rain_80 = 一小時降雨量 > 80  
cond_terrain_high = 地形風險 in [high, extremely_high]

# 風險分級
if 雨量 > 80 → CRITICAL
elif 雨量 > 40 and 地形高風險 → URGENT
elif 雨量 > 40 or 地形高風險 → WARNING
else → SAFE

In [ ]:
# 先建立預設欄位
gdf_shelter_buffer_risk_all["risk"] = "SAFE"

# 條件
cond_rain_80 = gdf_shelter_buffer_risk_all["一小時降雨量"] > 80
cond_rain_40 = gdf_shelter_buffer_risk_all["一小時降雨量"] > 40
cond_terrain_high = gdf_shelter_buffer_risk_all["terrain_risk"].isin(
    ["high", "extremely_high"]
)

# WARNING：時雨量 > 40 或 terrain_risk 為 high / extremely_high
gdf_shelter_buffer_risk_all.loc[
    cond_rain_40 | cond_terrain_high,
    "risk"
] = "WARNING"

# URGENT：時雨量 > 40 且 terrain_risk 為 high / extremely_high
gdf_shelter_buffer_risk_all.loc[
    cond_rain_40 & cond_terrain_high,
    "risk"
] = "URGENT"

# CRITICAL：時雨量 > 80
gdf_shelter_buffer_risk_all.loc[
    cond_rain_80,
    "risk"
] = "CRITICAL"

# 檢查結果
print(gdf_shelter_buffer_risk_all["risk"].value_counts(dropna=False))
display(gdf_shelter_buffer_risk_all.head())

# 輸出 CSV
import os

output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)

csv_path = f"{output_dir}/gdf_shelter_buffer_risk.csv"

export_df = gdf_shelter_buffer_risk_all.copy()
export_df["geometry"] = export_df["geometry"].astype(str)

export_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"✅ 已輸出：{csv_path}")

## Cell 15 - ARIA v3.0 互動式地圖

### 🎯 功能
建立完整的互動式地圖，整合雨量站、避難所、風險分級等所有資訊。

### 🗺️ 地圖圖層
- **雨量站點** - CircleMarker 依雨量大小顏色
- **雨量熱力圖** - HeatMap 顯示雨量分佈
- **避難所叢集** - MarkerCluster 優化顯示
- **風險分層** - 依 CRITICAL/URGENT/WARNING/SAFE 分層

### 🎨 視覺化特色
- **自定義叢集圖示** - 依數量變色的大小
- **動態圓形半徑** - 雨量越大圓形越大
- **豐富 Popup 資訊** - 完整的避難所與雨量站資訊
- **多種底圖選項** - CartoDB positron

### 📊 資料整合
- **避難所資料** - 地形風險 + 動態風險
- **雨量站資料** - 即時或模擬雨量資料
- **空間分析** - 緩衝區相交分析結果
- **綜合評估** - 多因子風險評估結果

### 🔄 輸出成果
- **HTML 地圖** - `ARIA_v3_Fungwong.html`
- **多圖層控制** - 可開關各種圖層
- **互動式查詢** - 點擊查看詳細資訊

### 🎯 使用方式
執行後會開啟包含所有分析結果的互動式地圖，可用來：
- 即時災害監測
- 避難所風險評估
- 緊急應變決策支援

In [ ]:
import folium
from folium.plugins import HeatMap
from folium.plugins import HeatMap, MarkerCluster

from folium.plugins import MarkerCluster

cluster_js = """
function(cluster) {
    var count = cluster.getChildCount();

    var fillColor = '#bdbdbd';
    var outerColor = 'rgba(189,189,189,0.35)';


    if (count >= 50) {
        fillColor = '#7f7f7f';
        outerColor = 'rgba(127,127,127,0.35)';
    } else if (count >= 25) {
        fillColor = '#a6a6a6';
        outerColor = 'rgba(166,166,166,0.35)';
    } else {
        fillColor = '#cfcfcf';
        outerColor = 'rgba(207,207,207,0.35)';
    }

    return L.divIcon({
        html: `
            <div style="
                background:${outerColor};
                width:40px;
                height:40px;
                border-radius:50%;
                display:flex;
                align-items:center;
                justify-content:center;
            ">
                <div style="
                    background:${fillColor};
                    width:30px;
                    height:30px;
                    border-radius:50%;
                    color:#222;
                    font-weight:bold;
                    font-size:14px;
                    line-height:30px;
                    text-align:center;
                ">${count}</div>
            </div>
        `,
        className: 'marker-cluster-custom',
        iconSize: new L.Point(40, 40)
    });
}
"""

shelter_cluster_layer = folium.FeatureGroup(name="Shelter Cluster", show=True)
cluster = MarkerCluster(
    icon_create_function=cluster_js
).add_to(shelter_cluster_layer)

# =========================================
# 1. 準備資料
# =========================================
# 避難所資料：gdf_shelter_buffer_risk_all
# 雨量站資料：gdf_rainfall_3826 或 gdf_fungwong
# 建議這裡使用已清理過 -998、且有 rain_1hr 的雨量站資料

# ---- 避難所：用經緯度欄位畫點 ----
shelter_df = gdf_shelter_buffer_risk_all.copy()
shelter_df["經度"] = pd.to_numeric(shelter_df["經度"], errors="coerce")
shelter_df["緯度"] = pd.to_numeric(shelter_df["緯度"], errors="coerce")
shelter_df["一小時降雨量"] = pd.to_numeric(shelter_df["一小時降雨量"], errors="coerce").fillna(0)
shelter_df = shelter_df.dropna(subset=["經度", "緯度"]).copy()

# ---- 雨量站：轉成 4326 畫 Folium ----
rain_df = gdf_rainfall_3826.copy()
if str(rain_df.crs) != "EPSG:4326":
    rain_df = rain_df.to_crs("EPSG:4326")

rain_df["lon"] = rain_df.geometry.x
rain_df["lat"] = rain_df.geometry.y
rain_df["rain_1hr"] = pd.to_numeric(rain_df["rain_1hr"], errors="coerce").fillna(0)
rain_df = rain_df[rain_df["rain_1hr"] != -998].copy()

# =========================================
# 2. 地圖中心（設在花蓮）
# =========================================
# 若你有 hualien_boundary，可優先用它中心點
if "hualien_boundary" in globals():
    hualien_4326 = hualien_boundary.to_crs("EPSG:4326")
    center_lon = hualien_4326.geometry.centroid.x.iloc[0]
    center_lat = hualien_4326.geometry.centroid.y.iloc[0]
else:
    center_lat = 23.9
    center_lon = 121.6

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=9,
    tiles="CartoDB positron"
)

# =========================================
# 3. 顏色函式
# =========================================
def rainfall_color(v):
    if v <= 10:
        return "green"
    elif v <= 30:
        return "yellow"
    elif v <= 80:
        return "orange"
    else:
        return "red"

def rainfall_radius(v):
    # 半徑依雨量放大，最小 4
    return max(4, min(18, 4 + v * 0.12))

def shelter_color(risk):
    risk = str(risk).upper()
    if risk == "CRITICAL":
        return "darkred"
    elif risk == "URGENT":
        return "red"
    elif risk == "WARNING":
        return "orange"
    else:
        return "green"

# =========================================
# 4. 雨量觀測站 CircleMarker 圖層（獨立 Layer）
#    透明度 20%
# =========================================
rain_station_layer = folium.FeatureGroup(name="Rainfall Stations", show=True)

for _, row in rain_df.iterrows():
    popup_html = f"""
    <div style="width:260px;">
        <h4 style="margin-bottom:8px;">{row.get('StationName', 'N/A')}</h4>
        <b>Station ID：</b>{row.get('StationId', 'N/A')}<br>
        <b>County：</b>{row.get('CountyName', 'N/A')}<br>
        <b>Town：</b>{row.get('TownName', 'N/A')}<br>
        <b>1hr Rainfall：</b>{row.get('rain_1hr', 'N/A')} mm/hr<br>
        <b>3hr Rainfall：</b>{row.get('rain_3hr', 'N/A')} mm<br>
        <b>24hr Rainfall：</b>{row.get('rain_24hr', 'N/A')} mm
    </div>
    """

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=rainfall_radius(row["rain_1hr"]),
        color=rainfall_color(row["rain_1hr"]),
        weight=1,
        fill=True,
        fill_color=rainfall_color(row["rain_1hr"]),
        fill_opacity=0.2,   # 20%
        opacity=0.2,        # 線條透明度也一起調低
        popup=folium.Popup(popup_html, max_width=320),
        tooltip=f"{row.get('StationName', 'N/A')} | {row.get('rain_1hr', 0)} mm/hr"
    ).add_to(rain_station_layer)

rain_station_layer.add_to(m)

# =========================================
# 5. HeatMap 圖層
# =========================================
heat_layer = folium.FeatureGroup(name="Rainfall HeatMap", show=False)

heat_data = rain_df[["lat", "lon", "rain_1hr"]].values.tolist()

HeatMap(
    heat_data,
    radius=25,
    blur=18,
    min_opacity=0.35,
    max_zoom=12
).add_to(heat_layer)

heat_layer.add_to(m)

# =========================================
# 6. 避難所圖層 + MarkerCluster
# =========================================
shelter_layer = folium.FeatureGroup(name="Shelters", show=True)
shelter_cluster = MarkerCluster(
    name="Shelter Cluster",
    icon_create_function=cluster_js
).add_to(shelter_layer)

for _, row in shelter_df.iterrows():
    popup_html = f"""
    <div style="width:280px;">
        <h4 style="margin-bottom:8px;">{row.get('避難收容處所名稱', 'N/A')}</h4>
        <b>地址：</b>{row.get('避難收容處所地址', 'N/A')}<br>
        <b>W4 地形風險：</b>{row.get('terrain_risk', 'N/A')}<br>
        <b>動態風險等級：</b>{row.get('risk', 'N/A')}<br>
        <b>一小時降雨量：</b>{row.get('一小時降雨量', 'N/A')}<br>
        <b>最近雨量站名：</b>{row.get('StationName', 'N/A')}<br>
        <b>村里：</b>{row.get('村里', 'N/A')}<br>
        <b>預計收容村里：</b>{row.get('預計收容村里', 'N/A')}
    </div>
    """

    folium.CircleMarker(
        location=[row["緯度"], row["經度"]],
        radius=5,
        color=shelter_color(row.get("risk", "SAFE")),
        fill=True,
        fill_color=shelter_color(row.get("risk", "SAFE")),
        fill_opacity=0.85,
        weight=1.5,
        popup=folium.Popup(popup_html, max_width=320),
        tooltip=f"{row.get('避難收容處所名稱', 'N/A')} | {row.get('risk', 'SAFE')}"
    ).add_to(shelter_cluster)

shelter_layer.add_to(m)

# =========================================
# 7. 可選：依風險分開圖層
# =========================================
for level in ["CRITICAL", "URGENT", "WARNING", "SAFE"]:
    fg = folium.FeatureGroup(name=f"Shelter Risk: {level}", show=False)
    sub = shelter_df[shelter_df["risk"].astype(str).str.upper() == level]

    for _, row in sub.iterrows():
        popup_html = f"""
        <div style="width:280px;">
            <h4 style="margin-bottom:8px;">{row.get('避難收容處所名稱', 'N/A')}</h4>
            <b>地址：</b>{row.get('避難收容處所地址', 'N/A')}<br>
            <b>W4 地形風險：</b>{row.get('terrain_risk', 'N/A')}<br>
            <b>動態風險等級：</b>{row.get('risk', 'N/A')}<br>
            <b>一小時降雨量：</b>{row.get('一小時降雨量', 'N/A')}<br>
            <b>最近雨量站名：</b>{row.get('StationName', 'N/A')}
        </div>
        """

        folium.CircleMarker(
            location=[row["緯度"], row["經度"]],
            radius=7,
            color=shelter_color(level),
            fill=True,
            fill_color=shelter_color(level),
            fill_opacity=0.95,
            popup=folium.Popup(popup_html, max_width=320),
            tooltip=f"{row.get('避難收容處所名稱', 'N/A')} | {level}"
        ).add_to(fg)

    fg.add_to(m)

# =========================================
# 8. LayerControl
# =========================================
folium.LayerControl(collapsed=False).add_to(m)

# =========================================
# 9. 儲存
# =========================================
output_html = "./output/ARIA_v3_Fungwong.html"
m.save(output_html)

print(f"✅ 地圖已輸出：{output_html}")

m